# Field Scan : détecteur de bananiers (DeepForest)

Apprendre à DeepForest, déjà entraîné à trouver des couronnes d'arbres sur images drone, à trouver **les bananiers**.

| Rôle | Fermes ReforesTree | Utilisation ici |
| --- | --- | --- |
| Entraînement | Flora Pluas, Leonor Aspiazu, Carlos Vera Guevara, Carlos Vera Arteaga | apprentissage |
| Validation | Manuel Macias | choix des réglages et note |
| Test | Nestor Macias | **non utilisé ici** (gardé pour la fin) |

Règle de notation du sujet : une détection est juste si son centre est à moins de **1 m** du centre d'un vrai bananier.

Avant de lancer : **Exécution → Modifier le type d'exécution → GPU T4**.
Données attendues : `MyDrive/field_scan/tuiles.zip` (dossier `tuiles` produit par `outils/preparer_tuiles_comptage.py`).

In [ ]:
# 1. Installation de DeepForest, accès au Drive, décompression des tuiles
!pip install -q "deepforest>=2,<3"
from google.colab import drive
drive.mount('/content/drive')
DOSSIER_DRIVE = '/content/drive/MyDrive/field_scan'
!unzip -q -o "{DOSSIER_DRIVE}/tuiles.zip" -d /content/
!ls /content/tuiles

In [ ]:
# 2. Réglages et lecture des tuiles
import json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import deepforest
from deepforest import main
from scipy.optimize import linear_sum_assignment

RACINE = Path('/content/tuiles')
IMAGES = RACINE / 'images'
ESSAI = 'essai1'                               # nom de l'essai : sert à nommer les fichiers sauvegardés
EPOQUES = 30
PART_TUILES_VIDES = 0.10                       # entraînement : les tuiles sans bananier ne dépassent pas 10 % des tuiles
TAILLE_LOT = 8
RAYON_M = 1.0                                  # règle du sujet
SEUILS_CONFIANCE = np.round(np.arange(0.05, 0.96, 0.05), 2)
SEUILS_NMS = [0.05, 0.2, 0.4]                  # recouvrement maximal entre deux détections gardées
LABEL = 'Tree'                                 # le modèle pré-entraîné a une seule classe, nommée « Tree » :
                                               # on la garde et on n'y met que les bananiers
print('DeepForest', deepforest.__version__, '| GPU :', torch.cuda.is_available())

tuiles = pd.read_csv(RACINE / 'tuiles.csv')
annotations = pd.read_csv(RACINE / 'annotations.csv')
if 'taille_pixel_m' not in tuiles.columns:
    raise ValueError("tuiles.csv doit contenir la colonne taille_pixel_m : relancer preparer_tuiles_comptage.py")
print(tuiles.groupby('role').agg(tuiles=('image_path', 'size'), bananes=('bananes', 'sum')))

In [ ]:
# 3. Fichiers d'annotations au format DeepForest (bananiers seulement)
#    Les autres arbres restent visibles sur l'image, sans rectangle : le modèle apprend qu'ils ne sont pas des bananiers.
#    Une tuile sans bananier est décrite par une ligne « vide » (0, 0, 0, 0), comme le demande DeepForest.
def fichier_deepforest(role, chemin):
    noms = tuiles[tuiles.role == role].image_path
    bananes = annotations[(annotations.label == 'banane') & annotations.image_path.isin(noms)].copy()
    bananes = bananes[(bananes.xmax > bananes.xmin) & (bananes.ymax > bananes.ymin)]
    bananes['label'] = LABEL
    noms_vides = sorted(set(noms) - set(bananes.image_path))
    if role == 'entrainement':
        # Trop de tuiles vides apprennent au modèle que « la plupart du temps il n'y a rien » :
        # on en garde au plus PART_TUILES_VIDES du total, tirées au hasard (graine fixe)
        maximum = int(PART_TUILES_VIDES * len(set(bananes.image_path)) / (1 - PART_TUILES_VIDES))
        print(f'{role:13s} {len(noms_vides)} tuiles sans bananier, {min(maximum, len(noms_vides))} gardées')
        noms_vides = list(pd.Series(noms_vides).sample(min(maximum, len(noms_vides)), random_state=0))
    vides = pd.DataFrame({'image_path': noms_vides, 'xmin': 0, 'ymin': 0, 'xmax': 0, 'ymax': 0, 'label': LABEL})
    df = pd.concat([bananes[['image_path', 'xmin', 'ymin', 'xmax', 'ymax', 'label']], vides], ignore_index=True)
    df.to_csv(chemin, index=False)
    print(f'{role:13s} {df.image_path.nunique():4d} tuiles utilisées, {len(bananes):5d} rectangles, {len(vides):3d} tuiles sans bananier')
    return chemin

csv_entrainement = fichier_deepforest('entrainement', RACINE / 'deepforest_entrainement.csv')
csv_validation = fichier_deepforest('validation', RACINE / 'deepforest_validation.csv')

In [ ]:
# 4. Notation à 1 m sur les tuiles de validation
#    Pour chaque tuile : centres des détections contre centres des vrais bananiers, appariement un pour un.
validation = tuiles[tuiles.role == 'validation']
verite = annotations[(annotations.label == 'banane') & annotations.image_path.isin(validation.image_path)]

def centres(df):
    return np.c_[(df.xmin + df.xmax) / 2, (df.ymin + df.ymax) / 2]

def apparier(det, vrai, rayon_px):
    if len(det) == 0 or len(vrai) == 0:
        return 0
    d = np.linalg.norm(det[:, None] - vrai[None], axis=2)
    cout = np.where(d <= rayon_px, d, 1e6)
    l, c = linear_sum_assignment(cout)
    return int((cout[l, c] <= rayon_px).sum())

def predire_validation(modele, nms):
    modele.model.nms_thresh = nms
    modele.model.score_thresh = 0.05          # on garde tout, le seuil est appliqué ensuite
    resultats = {}
    for nom in validation.image_path:
        p = modele.predict_image(path=str(IMAGES / nom))
        resultats[nom] = p if p is not None else pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'score'])
    return resultats

def noter(predictions, seuil):
    vp = nb_det = 0
    for _, t in validation.iterrows():
        p = predictions[t.image_path]
        p = p[p.score >= seuil]
        vrai = verite[verite.image_path == t.image_path]
        vp += apparier(centres(p), centres(vrai), RAYON_M / t.taille_pixel_m)
        nb_det += len(p)
    nb_vrais = len(verite)
    precision = vp / nb_det if nb_det else 0.0
    rappel = vp / nb_vrais if nb_vrais else 0.0
    f1 = 2 * precision * rappel / (precision + rappel) if precision + rappel else 0.0
    return {'detections': nb_det, 'bananes_vraies': nb_vrais, 'vp': vp, 'precision': precision,
            'rappel': rappel, 'f1': f1, 'erreur_comptage': (nb_det - nb_vrais) / nb_vrais}

def meilleur_reglage(modele, nom):
    lignes = []
    for nms in SEUILS_NMS:
        pred = predire_validation(modele, nms)
        for s in SEUILS_CONFIANCE:
            lignes.append({'modele': nom, 'nms': nms, 'seuil': s} | noter(pred, s))
    tableau = pd.DataFrame(lignes)
    meilleur = tableau.loc[tableau.f1.idxmax()]
    print(f"{nom} : meilleure F1 validation = {meilleur.f1:.3f} (nms {meilleur.nms}, seuil {meilleur.seuil}, "
          f"{int(meilleur.detections)} détections pour {int(meilleur.bananes_vraies)} bananiers)")
    return tableau, meilleur

In [ ]:
# 5. Avant apprentissage : le modèle « arbres » de DeepForest, tel quel
#    Il trouve des arbres, sans distinguer les bananiers : c'est le point de départ.
avant = main.deepforest()
avant.load_model('weecology/deepforest-tree')
tableau_avant, meilleur_avant = meilleur_reglage(avant, 'avant apprentissage')

In [ ]:
# 6. Apprentissage des bananiers, à partir du modèle « arbres »
modele = main.deepforest()
modele.load_model('weecology/deepforest-tree')
modele.config.train.csv_file = str(csv_entrainement)
modele.config.train.root_dir = str(IMAGES)
modele.config.validation.csv_file = str(csv_validation)
modele.config.validation.root_dir = str(IMAGES)
modele.config.train.epochs = EPOQUES
modele.config.batch_size = TAILLE_LOT
modele.config.workers = 2
torch.manual_seed(0)

# Journal : l'erreur est enregistrée à chaque époque, pour vérifier que le modèle apprend
try:
    from lightning.pytorch.loggers import CSVLogger
except ImportError:
    from pytorch_lightning.loggers import CSVLogger
journal = CSVLogger('/content/journaux', name=ESSAI)

debut = time.time()
modele.create_trainer(logger=journal)
modele.trainer.fit(modele)
print(f'Apprentissage terminé en {(time.time() - debut) / 60:.1f} min')

# Erreur moyenne par époque (elle doit baisser ; les colonnes « val_ » sont calculées sur la validation)
metriques = pd.read_csv(Path(journal.log_dir) / 'metrics.csv')
colonnes = [col for col in metriques.columns if 'loss' in col or col.startswith('val')]
print(metriques.groupby('epoch')[colonnes].mean().round(4).to_string())

In [ ]:
# 7. Après apprentissage : réglages (nms, seuil) choisis sur la validation, puis note
tableau_apres, meilleur_apres = meilleur_reglage(modele, 'après apprentissage')

# Confiance du modèle : un détecteur qui a bien appris donne des scores élevés sur ce qu'il reconnaît
scores = pd.concat(predire_validation(modele, float(meilleur_apres.nms)).values()).score
print(f'Scores des détections en validation : médiane {scores.median():.2f}, '
      f'90 % sont sous {scores.quantile(0.9):.2f}, maximum {scores.max():.2f}')
print()
print('Rappel, méthode simple (ExG + maxima locaux) sur Manuel Macias : F1 = 0,205')
print(f'DeepForest avant apprentissage                                 : F1 = {meilleur_avant.f1:.3f}')
print(f'DeepForest après apprentissage ({ESSAI})                        : F1 = {meilleur_apres.f1:.3f}')

In [ ]:
# 8. Sauvegarde sur le Drive : poids, réglages retenus, tableaux complets
sortie = Path(DOSSIER_DRIVE) / 'resultats_comptage'
sortie.mkdir(exist_ok=True)
torch.save(modele.model.state_dict(), sortie / f'deepforest_bananiers_{ESSAI}.pt')
pd.concat([tableau_avant, tableau_apres]).to_csv(sortie / f'validation_reglages_{ESSAI}.csv', index=False)
metriques.to_csv(sortie / f'journal_{ESSAI}.csv', index=False)
json.dump({'essai': ESSAI, 'deepforest': deepforest.__version__, 'epoques': EPOQUES, 'taille_lot': TAILLE_LOT,
           'part_tuiles_vides': PART_TUILES_VIDES,
           'avant': meilleur_avant.to_dict(), 'apres': meilleur_apres.to_dict()},
          open(sortie / f'resultats_{ESSAI}.json', 'w'), indent=2, ensure_ascii=False, default=float)
print('Écrit dans', sortie)

## Diagnostic (sans réentraîner)

Pour relancer seulement le diagnostic : exécuter les cellules 1 à 4, puis directement les cellules 9 et 10.
Le modèle entraîné est rechargé depuis le Drive (`resultats_comptage/deepforest_bananiers_<essai>.pt`).

In [ ]:
# 9. Où sont les erreurs ? Note selon la distance acceptée, et distance au point le plus proche
#    Les points du modèle ne bougent pas : on change seulement la tolérance de la règle,
#    pour savoir si le modèle « voit » les bananiers mais pose son point à côté.
if 'modele' not in globals():
    modele = main.deepforest()
    modele.load_model('weecology/deepforest-tree')
    poids = Path(DOSSIER_DRIVE) / 'resultats_comptage' / f'deepforest_bananiers_{ESSAI}.pt'
    modele.model.load_state_dict(torch.load(poids, map_location='cpu'))
if 'meilleur_apres' not in globals():
    meilleur_apres = pd.Series(json.load(open(Path(DOSSIER_DRIVE) / 'resultats_comptage' / f'resultats_{ESSAI}.json'))['apres'])
NMS, SEUIL = float(meilleur_apres.nms), float(meilleur_apres.seuil)
predictions = predire_validation(modele, NMS)

def f1_rayon(rayon_m):
    vp = nb_det = 0
    for _, t in validation.iterrows():
        p = predictions[t.image_path]; p = p[p.score >= SEUIL]
        vrai = verite[verite.image_path == t.image_path]
        vp += apparier(centres(p), centres(vrai), rayon_m / t.taille_pixel_m)
        nb_det += len(p)
    precision, rappel = vp / max(nb_det, 1), vp / len(verite)
    return {'rayon_m': rayon_m, 'justes': vp, 'precision': round(precision, 3), 'rappel': round(rappel, 3),
            'f1': round(2 * precision * rappel / (precision + rappel), 3) if vp else 0.0}

print(f'Réglages : nms {NMS}, seuil {SEUIL}')
print(pd.DataFrame([f1_rayon(r) for r in [1.0, 1.5, 2.0, 3.0]]).to_string(index=False))

# Pour chaque vrai bananier : distance (en m) jusqu'à la détection la plus proche
distances = []
for _, t in validation.iterrows():
    p = predictions[t.image_path]; p = p[p.score >= SEUIL]
    vrai = verite[verite.image_path == t.image_path]
    if len(vrai) == 0:
        continue
    if len(p) == 0:
        distances += [np.inf] * len(vrai); continue
    d = np.linalg.norm(centres(vrai)[:, None] - centres(p)[None], axis=2).min(axis=1) * t.taille_pixel_m
    distances += list(d)
classes = pd.cut(pd.Series(distances), [0, 1, 2, 3, np.inf], right=False,
                 labels=['moins de 1 m', '1 à 2 m', '2 à 3 m', '3 m ou plus (rien autour)'])
print()
print('Distance entre chaque vrai bananier et la détection la plus proche :')
print(classes.value_counts().sort_index().to_string())

In [ ]:
# 10. Image : 6 tuiles de validation. Rouge = vrais bananiers (cercle de 1 m) ; bleu = détections du modèle (score)
from PIL import Image, ImageDraw
choix = validation.sort_values('bananes', ascending=False).head(6)
planche = Image.new('RGB', (3 * 410, 2 * 430), 'white')
dessin = ImageDraw.Draw(planche)
for k, (_, t) in enumerate(choix.iterrows()):
    im = Image.open(IMAGES / t.image_path).convert('RGB'); d = ImageDraw.Draw(im)
    rayon = 1.0 / t.taille_pixel_m
    for x, y in centres(verite[verite.image_path == t.image_path]):
        d.ellipse([x - rayon, y - rayon, x + rayon, y + rayon], outline='red', width=3)
    p = predictions[t.image_path]; p = p[p.score >= SEUIL]
    for _, b in p.iterrows():
        d.rectangle([b.xmin, b.ymin, b.xmax, b.ymax], outline='blue', width=2)
        d.text((b.xmin + 2, b.ymin + 1), f'{b.score:.2f}', fill='blue')
    X, Y = (k % 3) * 410, (k // 3) * 430
    planche.paste(im, (X, Y))
    dessin.text((X, Y + 403), f'{t.image_path} : {t.bananes} bananiers, {len(p)} détections', fill='black')
chemin = Path(DOSSIER_DRIVE) / 'resultats_comptage' / f'diagnostic_validation_{ESSAI}.png'
planche.save(chemin)
print('Image enregistrée :', chemin)
planche